# PHH Rank Similarity Analysis

PHH（Primary Human Hepatocytes）を正常基準として、glycogenesの「rank signature」を作成し、
各化合物（LINCS）のglycogene rank signatureがPHHにどれだけ近いか（Spearman相関）でスコア化する。

## 解析の流れ
1. 入力データの読み込み（PHH、LINCS、glycogene list）
2. PHH rank signatureの作成
3. 化合物ごとのrank signature作成
4. PHHへの類似度計算（Spearman相関 + Bootstrap CI）
5. ランキングと可視化
6. 解釈（寄与遺伝子の抽出）
7. レポート生成

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import logging
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kruskal
import warnings
warnings.filterwarnings('ignore')

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Project root settings (go up 2 levels from notebooks/analytics/)
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

# Module imports
import task2_phh_rank_similarity as phh
from utils.pathway_abbreviations import abbreviate_pathway

# Results directory (notebooks/results/phh_similarity)
results_dir = project_root / 'notebooks' / 'results' / 'phh_similarity'
results_dir.mkdir(parents=True, exist_ok=True)

# Input directory
inputs_dir = project_root / 'inputs'
inputs_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Results directory: {results_dir}')
print(f'Input directory: {inputs_dir}')

## Publication-quality Visualization Settings

In [ ]:
# Publication-quality settings (following figure-style guidelines)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    # Font sizes (following figure-style guidelines)
    'axes.labelsize': 18,      # Axis labels
    'axes.titlesize': 16,      # Titles
    'xtick.labelsize': 16,     # X-axis ticks
    'ytick.labelsize': 16,     # Y-axis ticks
    'legend.fontsize': 16,     # Legend
    'font.size': 16,           # Default
    
    # SVG output settings
    'svg.fonttype': 'none',    # Do not convert text to paths
    
    # Resolution/output
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.format': 'svg',
    'savefig.bbox': 'tight',
    
    # Other
    'axes.linewidth': 1.2,
    'lines.linewidth': 1.5,
    'grid.linewidth': 0.5,
})

# Random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Visualization settings complete (following figure-style guidelines)')

## Parameter Settings

In [ ]:
# Parameters
PARAMS = {
    'min_genes': 300,           # Minimum gene count (compounds below this are excluded)
    'bootstrap_n': 100,         # Bootstrap iteration count
    'bootstrap_fraction': 0.8,  # Gene fraction to sample in Bootstrap
    'rank_ascending': True,     # True: rank=1 for lowest z-score
    'n_top_compounds': 20,      # Number of top compounds for interpretation
    'n_genes_interpret': 30,    # Number of genes to extract for interpretation
}

print('Parameter settings:')
for key, value in PARAMS.items():
    print(f'  {key}: {value}')

## 1. Snowflake Connection Settings

In [ ]:
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect

def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH", "~/.ssh/snowflake_rsa_key.pem")')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user=os.environ.get("SNOWFLAKE_USER"),
            account=os.environ.get("SNOWFLAKE_ACCOUNT"),
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

print('Snowflake connection functions defined')

## 2. Load Input Data

### 2.1 Glycogene List

In [ ]:
# Load glycogene list
glycogenes_path = project_root / 'GlycoEnzOnto' / 'glycogenes_from_gmt.txt'

with open(glycogenes_path, 'r') as f:
    glycogenes_raw = [line.strip().strip('"').strip("'") for line in f if line.strip()]

glycogenes_list = sorted(list(set(glycogenes_raw)))

print(f'Number of glycogenes: {len(glycogenes_list)}')
print(f'Examples: {glycogenes_list[:10]}')

# Also save to inputs/glycogenes.txt (matching specification format)
glycogenes_file = inputs_dir / 'glycogenes.txt'
with open(glycogenes_file, 'w') as f:
    f.write('\n'.join(glycogenes_list))
print(f'\nSaved: {glycogenes_file}')

### 2.2 Liver Cancer Cell Line List

In [ ]:
# Liver cancer cell line list
liver_cells_path = project_root / 'cell_line_name' / 'results' / 'liver_cell_lines_list.csv'
df_liver_list = pd.read_csv(liver_cells_path)
cell_lines = df_liver_list['CELL_LINE_NAME'].tolist()

print(f'Liver cancer cell lines: {cell_lines}')

### 2.4 Retrieve LINCS Data (Snowflake)

In [ ]:
# Connect to Snowflake
conn = connect_to_snowflake()

# Get glycogene column names (verify Snowflake table structure)
metadata_columns = {
    'VALUE', 'canonical_smiles', 'cell', 'cmapid', 'compound_alias',
    'dose', 'inchi_key', 'pertid', 'pertname', 'timepoint', 'sample_id'
}

column_query = """
SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'LINCS'
AND TABLE_NAME = 'GLYCO_GENES_WIDE'
ORDER BY COLUMN_NAME
"""

column_df = pd.read_sql(column_query, conn)
all_columns = column_df['COLUMN_NAME'].tolist()
glyco_genes_db = [col for col in all_columns if col not in metadata_columns]

print(f'Snowflake glycogene count: {len(glyco_genes_db)}')

In [ ]:
# Retrieve LINCS data
gene_columns = ', '.join([f'"{gene}"' for gene in glyco_genes_db])
cell_lines_str = "', '".join(cell_lines)

query = f"""
SELECT
    "sample_id",
    "canonical_smiles",
    "inchi_key",
    "pertname",
    "pertid",
    "cell",
    "dose",
    "timepoint",
    {gene_columns}
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE
WHERE "cell" IN ('{cell_lines_str}')
"""

print('Executing query...')
df_lincs = pd.read_sql(query, conn)
print(f'LINCS data retrieval complete: {len(df_lincs):,} records')
print(f'Number of compounds: {df_lincs["pertname"].nunique()}')
print(f'Cell lines: {df_lincs["cell"].unique()}')

# Convert to numeric type
for gene in glyco_genes_db:
    if gene in df_lincs.columns:
        df_lincs[gene] = pd.to_numeric(df_lincs[gene], errors='coerce')

df_lincs.head()

### 2.3 Retrieve PHH Control Data (Snowflake)

Retrieve PHH (Primary Human Hepatocytes) DMSO control data from Snowflake.
- **Data source**: `CTL_GLYCO_GENES_WIDE` table (dedicated to control samples)
- **Condition**: DMSO treatment only (excluding UnTrt)
- **Timepoints**: Using combined 6h + 24h
- **Purpose**: Use as baseline signature for normal hepatocytes

In [ ]:
# Retrieve PHH DMSO control data from CTL_GLYCO_GENES_WIDE
query_phh = """
SELECT *
FROM BIOINFORMATICS.LINCS.CTL_GLYCO_GENES_WIDE
WHERE CELL = 'PHH'
  AND PERTNAME = 'DMSO'
"""

print('Retrieving PHH DMSO control data...')
df_phh_raw = pd.read_sql(query_phh, conn)
print(f'PHH data retrieval complete: {len(df_phh_raw):,} records')

# Convert column names to lowercase (for compatibility with existing code)
df_phh_raw.columns = df_phh_raw.columns.str.lower()

# Data quality check
print('\n[Data Quality Check]')
print(f'Timepoints: {df_phh_raw["timepoint"].unique()}')
print(f'PERTNAME: {df_phh_raw["pertname"].unique()}')

# Sample count by timepoint
print('\n[Sample Count by Timepoint]')
print(df_phh_raw.groupby('timepoint').size())

# No filtering needed since this is DMSO control
df_phh = df_phh_raw.copy()

print(f'\n[Final Data]')
print(f'  Sample count: {len(df_phh):,}')
print(f'  Timepoints: {sorted(df_phh["timepoint"].unique())}')

# Identify gene columns (excluding metadata)
metadata_cols = {'sample_id', 'cell', 'pertname', 'dose', 'timepoint'}
gene_cols_phh = [c for c in df_phh.columns if c not in metadata_cols]
print(f'  Number of genes: {len(gene_cols_phh)}')

# Convert to numeric type
for gene in gene_cols_phh:
    df_phh[gene] = pd.to_numeric(df_phh[gene], errors='coerce')

print(f'\nPHH data shape: {df_phh.shape}')
df_phh.head()

## 3. Create PHH Rank Signature

Create a rank signature from PHH data. For each sample, rank glycogene z-scores and take the median across samples to create a representative signature.

In [ ]:
print('=' * 60)
print('Creating PHH Rank Signature')
print('=' * 60)

# Match PHH gene columns (lowercase) with LINCS gene columns (uppercase)
# df_phh is lowercase, glyco_genes_db is uppercase, so convert to lowercase for comparison
glyco_genes_db_lower = [g.lower() for g in glyco_genes_db]
available_glyco = [g for g in gene_cols_phh if g.lower() in glyco_genes_db_lower]
print(f'\nAvailable glycogenes: {len(available_glyco)}')

# Create PHH rank signature
df_phh_sig = phh.create_phh_reference_signature(
    df_phh,
    available_glyco,
    sample_id_col='sample_id',
    ascending=PARAMS['rank_ascending']
)

# Convert gene names back to uppercase (for consistency with LINCS data)
df_phh_sig['gene'] = df_phh_sig['gene'].str.upper()

print(f'\nPHH Rank Signature:')
print(f'  Number of genes: {len(df_phh_sig)}')
print(f'  Samples used: {df_phh_sig["n_samples_used"].max()}')

# Save
df_phh_sig.to_parquet(results_dir / 'phh_rank_signature.parquet', index=False)
print(f'\nSaved: {results_dir / "phh_rank_signature.parquet"}')

df_phh_sig.head(10)

## 4. Create Compound Rank Signatures

In [ ]:
print('=' * 60)
print('Creating Compound Rank Signatures')
print('=' * 60)

# Use only glycogenes common with PHH
phh_genes = set(df_phh_sig['gene'].values)
lincs_genes = [g for g in glyco_genes_db if g in df_lincs.columns and g in phh_genes]
print(f'\nGlycogenes common with PHH: {len(lincs_genes)}')

# Create compound rank signatures
print('\nCreating compound rank signatures...')
df_compound_sig = phh.create_compound_signatures(
    df_lincs,
    lincs_genes,
    compound_id_col='pertname',
    inchi_key_col='inchi_key',
    sample_id_col='sample_id',
    ascending=PARAMS['rank_ascending']
)

print(f'\nCompound Rank Signatures:')
print(f'  Number of compounds: {df_compound_sig["compound_id"].nunique()}')
print(f'  Number of records: {len(df_compound_sig):,}')

# Save
df_compound_sig.to_parquet(results_dir / 'compound_rank_signatures.parquet', index=False)
print(f'\nSaved: {results_dir / "compound_rank_signatures.parquet"}')

df_compound_sig.head(10)

## 5. Calculate PHH Similarity

In [ ]:
print('=' * 60)
print('Calculating PHH Similarity')
print('=' * 60)

print(f'\nParameters:')
print(f'  Minimum genes: {PARAMS["min_genes"]}')
print(f'  Bootstrap iterations: {PARAMS["bootstrap_n"]}')
print(f'  Bootstrap sampling rate: {PARAMS["bootstrap_fraction"]*100:.0f}%')

print('\nCalculating similarity...')
df_similarity = phh.calculate_phh_similarity(
    df_compound_sig,
    df_phh_sig,
    min_genes=PARAMS['min_genes'],
    bootstrap_n=PARAMS['bootstrap_n'],
    bootstrap_fraction=PARAMS['bootstrap_fraction'],
    random_seed=RANDOM_SEED
)

# Sort
df_similarity = df_similarity.sort_values('similarity_spearman', ascending=False).reset_index(drop=True)

print(f'\nPHH Similarity Results:')
print(f'  Compounds analyzed: {len(df_similarity)}')
print(f'  Similarity range: [{df_similarity["similarity_spearman"].min():.4f}, {df_similarity["similarity_spearman"].max():.4f}]')
print(f'  Mean: {df_similarity["similarity_spearman"].mean():.4f}')
print(f'  Median: {df_similarity["similarity_spearman"].median():.4f}')

# Save
df_similarity.to_parquet(results_dir / 'phh_similarity_scores.parquet', index=False)
df_similarity.to_csv(results_dir / 'phh_similarity_scores.csv', index=False)
print(f'\nSaved: {results_dir / "phh_similarity_scores.parquet"}')

print('\nTop 10 Compounds (most similar to PHH):')
df_similarity.head(10)

## 6. Ranking and Visualization

### 6.1 Top 30 Compounds Bar Plot

In [ ]:
print('Creating Top 30 Compounds Bar Plot...')

fig = phh.plot_top_compounds_barplot(
    df_similarity,
    n_top=30,
    figsize=(12, 12),
    output_path=results_dir / 'fig_top30_similarity.png'
)
plt.show()

### 6.2 Similarity Distribution

In [ ]:
print('Creating similarity distribution plot...')

fig = phh.plot_similarity_distribution(
    df_similarity,
    figsize=(10, 6),
    output_path=results_dir / 'fig_similarity_distribution.png'
)
plt.show()

## 7. Interpretation (Extract Contributing Genes)

### Purpose
Identify **which genes contribute** to the PHH similarity score.

### Method
For each compound, calculate the **rank difference** (|rank_compound - rank_PHH|) for each gene.

- **Genes close to PHH**: Small rank difference -> The gene's rank matches PHH
- **Genes far from PHH**: Large rank difference -> The gene's rank deviates from PHH

### Interpretation
- **Frequently appearing PHH-close genes** = Genes that commonly match PHH pattern across high-similarity compounds
  -> These are candidate glycogenes that are "easy to normalize"
- **Frequently appearing PHH-distant genes** = Genes that don't match PHH even in high-similarity compounds
  -> Genes possibly constitutively dysregulated in liver cancer cell lines, or glycogenes difficult to control with drugs

### Output
- `top20_genes_closest_to_phh.csv`: Top 30 genes closest to PHH for each of the Top 20 compounds
- `top20_genes_farthest_from_phh.csv`: Top 30 genes farthest from PHH for each of the Top 20 compounds

In [ ]:
print('=' * 60)
print('Extracting Contributing Genes')
print('=' * 60)

# Top N compounds
top_compound_ids = df_similarity.head(PARAMS['n_top_compounds'])['compound_id'].tolist()
print(f'\nTarget compounds: Top {len(top_compound_ids)}')

# Extract contributing genes
df_closest, df_farthest = phh.identify_contributing_genes(
    df_compound_sig,
    df_phh_sig,
    top_compound_ids,
    n_genes=PARAMS['n_genes_interpret']
)

print(f'\nGenes most aligned: {len(df_closest)} records')
print(f'Genes least aligned: {len(df_farthest)} records')

# Save
df_closest.to_csv(results_dir / 'top20_genes_closest_to_phh.csv', index=False)
df_farthest.to_csv(results_dir / 'top20_genes_farthest_from_phh.csv', index=False)

print(f'\nSaved: {results_dir / "top20_genes_closest_to_phh.csv"}')
print(f'Saved: {results_dir / "top20_genes_farthest_from_phh.csv"}')

# Aggregation (frequently appearing genes across all Top compounds)
print('\n[Genes Closest to PHH (Frequent in Top 20 Compounds)]')
closest_gene_counts = df_closest['gene'].value_counts().head(20)
for gene, count in closest_gene_counts.items():
    print(f'  {gene}: ranked high in {count} compounds')

print('\n[Genes Farthest from PHH (Frequent in Top 20 Compounds)]')
farthest_gene_counts = df_farthest['gene'].value_counts().head(20)
for gene, count in farthest_gene_counts.items():
    print(f'  {gene}: ranked high in {count} compounds')

## 7.2 Pathway Enrichment Analysis of Contributing Genes

Perform enrichment analysis of PHH-similar and PHH-dissimilar gene sets against GlycoEnzOnto pathways.

- **PHH-similar genes**: Genes showing matching patterns with PHH in high-similarity compounds -> pathways related to normalization
- **PHH-dissimilar genes**: Genes deviating from PHH even in high-similarity compounds -> cancer-associated pathways

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

print('=' * 60)
print('Pathway Enrichment Analysis of Contributing Genes')
print('=' * 60)

# Load GlycoEnzOnto GMT file
gmt_file = project_root / 'GlycoEnzOnto' / 'GlycoEnzOnto.gmt'

pathways = {}
with open(gmt_file, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            pathway_name = parts[0]
            genes = set(parts[2:])
            pathways[pathway_name] = genes

all_glycogenes = set()
for genes in pathways.values():
    all_glycogenes.update(genes)

print(f'Number of pathways: {len(pathways)}')
print(f'Total glycogenes: {len(all_glycogenes)}')

# Extract genes with frequency >= 3 (using Top 30 compounds)
N_TOP_COMPOUNDS = 30
top_compound_ids_30 = df_similarity.head(N_TOP_COMPOUNDS)['compound_id'].tolist()

# Re-extract contributing genes for Top 30 compounds
df_closest_30, df_farthest_30 = phh.identify_contributing_genes(
    df_compound_sig,
    df_phh_sig,
    top_compound_ids_30,
    n_genes=20  # Top 20 genes from each compound
)

# Save
df_closest_30.to_csv(results_dir / 'top30_genes_closest_to_phh.csv', index=False)
df_farthest_30.to_csv(results_dir / 'top30_genes_farthest_from_phh.csv', index=False)

# Extract genes with frequency >= 3
threshold = 3
closest_freq = df_closest_30['gene'].value_counts()
farthest_freq = df_farthest_30['gene'].value_counts()

genes_closest = set(closest_freq[closest_freq >= threshold].index)
genes_farthest = set(farthest_freq[farthest_freq >= threshold].index)

print(f'\nGenes with frequency >= {threshold} (Top {N_TOP_COMPOUNDS} compounds):')
print(f'  PHH-similar: {len(genes_closest)}')
print(f'  PHH-dissimilar: {len(genes_farthest)}')

In [ ]:
# Enrichment analysis function
def enrichment_analysis(gene_set, pathways, background):
    """Pathway enrichment analysis using Fisher's exact test"""
    results = []
    gene_set = set(gene_set) & background
    
    for pathway_name, pathway_genes in pathways.items():
        pathway_genes = pathway_genes & background
        
        a = len(gene_set & pathway_genes)  # overlap
        b = len(gene_set - pathway_genes)  # gene_set only
        c = len(pathway_genes - gene_set)  # pathway only
        d = len(background - gene_set - pathway_genes)  # neither
        
        if a == 0:
            continue
            
        table = [[a, b], [c, d]]
        odds_ratio, pvalue = fisher_exact(table, alternative='greater')
        fold = (a / (a + b)) / ((a + c) / len(background)) if (a + b) > 0 and (a + c) > 0 else 0
        
        results.append({
            'pathway': pathway_name,
            'overlap': a,
            'gene_set_size': len(gene_set),
            'pathway_size': len(pathway_genes),
            'fold_enrichment': fold,
            'pvalue': pvalue,
            'genes': ','.join(sorted(gene_set & pathway_genes))
        })
    
    if not results:
        return pd.DataFrame()
    
    df = pd.DataFrame(results)
    _, qvalues, _, _ = multipletests(df['pvalue'], method='fdr_bh')
    df['qvalue'] = qvalues
    return df.sort_values('pvalue')

# Run enrichment analysis
print('\n=== Enrichment Analysis ===')

df_enrich_closest = enrichment_analysis(genes_closest, pathways, all_glycogenes)
df_enrich_farthest = enrichment_analysis(genes_farthest, pathways, all_glycogenes)

# Display results
print('\nPHH-similar gene enrichment (p < 0.05):')
if len(df_enrich_closest) > 0:
    sig_closest = df_enrich_closest[df_enrich_closest['pvalue'] < 0.05]
    for _, row in sig_closest.head(10).iterrows():
        print(f"  {row['pathway'][:50]}: fold={row['fold_enrichment']:.2f}, p={row['pvalue']:.4f}")
    if len(sig_closest) == 0:
        print("  (No results with p < 0.05)")

print('\nPHH-dissimilar gene enrichment (p < 0.05):')
if len(df_enrich_farthest) > 0:
    sig_farthest = df_enrich_farthest[df_enrich_farthest['pvalue'] < 0.05]
    for _, row in sig_farthest.head(10).iterrows():
        print(f"  {row['pathway'][:50]}: fold={row['fold_enrichment']:.2f}, p={row['pvalue']:.4f}")
    if len(sig_farthest) == 0:
        print("  (No results with p < 0.05)")

# Combine and save results
df_enrich_closest['direction'] = 'PHH_similar'
df_enrich_farthest['direction'] = 'PHH_dissimilar'
df_combined = pd.concat([df_enrich_closest, df_enrich_farthest])
df_combined.to_csv(results_dir / 'phh_contributing_genes_pathway_top30.csv', index=False)
print(f'\nSaved: {results_dir / "phh_contributing_genes_pathway_top30.csv"}')

In [ ]:
# Pathway enrichment visualization
print('Visualizing pathway enrichment...')

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# PHH-similar genes
ax = axes[0]
if len(df_enrich_closest) > 0:
    plot_data = df_enrich_closest[df_enrich_closest['pvalue'] < 0.1].head(15)
    if len(plot_data) > 0:
        colors = ['darkred' if p < 0.01 else 'red' if p < 0.05 else 'salmon' 
                  for p in plot_data['pvalue']]
        y_pos = range(len(plot_data))
        ax.barh(y_pos, -np.log10(plot_data['pvalue']), color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([abbreviate_pathway(p)[:45] for p in plot_data['pathway']], fontsize=10)
        ax.set_xlabel(r'$-\log_{10}(p\text{-value})$', fontsize=10)
        ax.set_title('PHH-similar genes\n(Normalization pathways)', fontsize=12)
        ax.tick_params(axis='x', labelsize=10)
        ax.invert_yaxis()
        ax.axvline(x=-np.log10(0.05), color='gray', linestyle='--', linewidth=0.8, label='p=0.05')
        for i, (_, row) in enumerate(plot_data.iterrows()):
            ax.text(-np.log10(row['pvalue']) + 0.05, i, f"n={row['overlap']}", va='center', fontsize=7)
    else:
        ax.text(0.5, 0.5, 'No significant results (p < 0.1)', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('PHH-similar genes', fontsize=12)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

# PHH-dissimilar genes
ax = axes[1]
if len(df_enrich_farthest) > 0:
    plot_data = df_enrich_farthest[df_enrich_farthest['pvalue'] < 0.1].head(15)
    if len(plot_data) > 0:
        colors = ['darkblue' if p < 0.01 else 'blue' if p < 0.05 else 'lightblue' 
                  for p in plot_data['pvalue']]
        y_pos = range(len(plot_data))
        ax.barh(y_pos, -np.log10(plot_data['pvalue']), color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([abbreviate_pathway(p)[:45] for p in plot_data['pathway']], fontsize=10)
        ax.set_xlabel(r'$-\log_{10}(p\text{-value})$', fontsize=10)
        ax.set_title('PHH-dissimilar genes\n(Cancer-associated pathways)', fontsize=12)
        ax.tick_params(axis='x', labelsize=10)
        ax.invert_yaxis()
        ax.axvline(x=-np.log10(0.05), color='gray', linestyle='--', linewidth=0.8, label='p=0.05')
        for i, (_, row) in enumerate(plot_data.iterrows()):
            ax.text(-np.log10(row['pvalue']) + 0.05, i, f"n={row['overlap']}", va='center', fontsize=7)
    else:
        ax.text(0.5, 0.5, 'No significant results (p < 0.1)', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('PHH-dissimilar genes', fontsize=12)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(results_dir / 'phh_contributing_genes_pathway_top30.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'phh_contributing_genes_pathway_top30.svg', format='svg', bbox_inches='tight')
print(f'Saved: {results_dir / "phh_contributing_genes_pathway_top30.png"}')
plt.show()

In [ ]:
# Contributing gene frequency plot (Top 30 compounds)
print('Creating contributing gene frequency plot...')

fig, axes = plt.subplots(1, 2, figsize=(14, 10))

# PHH-similar genes (top 20 by frequency)
ax = axes[0]
top_closest = closest_freq.head(20)
colors = ['darkred' if f >= 10 else 'red' if f >= 5 else 'salmon' for f in top_closest.values]
y_pos = range(len(top_closest))
ax.barh(y_pos, top_closest.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_closest.index, style='italic', fontsize=16)
ax.set_xlabel(f'Frequency (out of {N_TOP_COMPOUNDS} compounds)')
ax.set_title('PHH-similar genes\n(Genes matching PHH pattern)')
ax.invert_yaxis()
for i, v in enumerate(top_closest.values):
    ax.text(v + 0.3, i, str(v), va='center')

# PHH-dissimilar genes (top 20 by frequency)
ax = axes[1]
top_farthest = farthest_freq.head(20)
colors = ['darkblue' if f >= 10 else 'blue' if f >= 5 else 'lightblue' for f in top_farthest.values]
y_pos = range(len(top_farthest))
ax.barh(y_pos, top_farthest.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_farthest.index, style='italic', fontsize=16)
ax.set_xlabel(f'Frequency (out of {N_TOP_COMPOUNDS} compounds)')
ax.set_title('PHH-dissimilar genes\n(Genes deviating from PHH pattern)')
ax.invert_yaxis()
for i, v in enumerate(top_farthest.values):
    ax.text(v + 0.3, i, str(v), va='center')

plt.tight_layout()
plt.savefig(results_dir / 'phh_contributing_genes_frequency_top30.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'phh_contributing_genes_frequency_top30.svg', format='svg', bbox_inches='tight')
print(f'Saved: {results_dir / "phh_contributing_genes_frequency_top30.png"}')
plt.show()

## 8. Generate Report

In [ ]:
print('=' * 60)
print('Generating Report')
print('=' * 60)

report = phh.generate_report(
    df_similarity,
    df_phh_sig,
    df_compound_sig,
    params=PARAMS,
    output_path=results_dir / 'report.md'
)

print(report)

## 9. Cleanup

In [ ]:
# Close Snowflake connection
conn.close()
print('Snowflake connection closed')

print('\n' + '=' * 60)
print('Output Files')
print('=' * 60)
for f in sorted(results_dir.glob('*')):
    print(f'  {f.name}')

print('\n' + '=' * 60)
print('All processing complete!')
print('=' * 60)